In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import numpy as np
from model import WaveFunctionNN, train, predicted
from prepare_data import get_dataloader_from_csv_file, extract_t_and_y_from_loader
from draw_with_plot import plot_data_and_predicted, draw_t_distribution
import pandas as pd

In [ ]:
SEED = 42
BATCH_SIZE = 100
HIDDEN_SIZE = 128
NUM_LAYERS = 3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 400

LR = 0.05

torch.manual_seed(42)
print(f"device: {DEVICE}")

In [ ]:
train_loader, eval_loader, scaler = get_dataloader_from_csv_file('finance.csv', ['t'], ['y'], BATCH_SIZE)

model = WaveFunctionNN(HIDDEN_SIZE, NUM_LAYERS)

best_loss = train(model, train_loader, eval_loader, EPOCHS, LR, DEVICE, eval_method='grid')


In [ ]:
model.load_state_dict(torch.load('models_pth/best_model.pth'))

t, original_x = extract_t_and_y_from_loader(eval_loader)

predicted_t = torch.tensor(t.reshape(-1, 1)).to(DEVICE)
predicted_x = predicted(model, predicted_t, DEVICE)

predicted_t = predicted_t.cpu().numpy()
predicted_x = scaler.inverse_transform(predicted_x.cpu().numpy())

original_x = scaler.inverse_transform(original_x.reshape(-1, 1))

predicted_data = pd.DataFrame({'t': predicted_t.reshape(-1), 'y': predicted_x.reshape(-1)})
original_data = pd.DataFrame({'t': t.reshape(-1), 'y': original_x.reshape(-1)})

plot_data_and_predicted("true and predicted", original_data, predicted_data)

In [ ]:
predicted_data.to_csv('predicted_with_no_schrodinger.csv', index=False)